In [ ]:
%matplotlib widget

In [ ]:
from pathlib import Path
import numpy as np
import flammkuchen as fl
import pandas as pd
import tifffile as tiff

from fimpylab import TwoPExperiment
from split_dataset import SplitDataset

import matplotlib.pyplot as plt
import json

from scipy.interpolate import interp1d
from scipy import signal
from lotr.default_vals import REGRESSOR_TAU_S, TURN_BIAS

In [ ]:
def exp_decay_kernel(tau, dt, len_rec):
    upsample = 10
    t = np.arange(len_rec * upsample) * dt / upsample
    
    decay = np.exp(-t / tau)
    decay /= np.sum(decay)
    return decay

In [ ]:
master = Path(r"Z:\Hagar\e0075\v06 and v09")
files = list(master.glob("*_f*"))

In [ ]:
num_rows = 9
s_size = 1 / num_rows
choices = []
for x_pos in range(num_rows):
    for y_pos in range(num_rows):
        curr_choice = [x_pos / num_rows, y_pos / num_rows, s_size, s_size]
        choices.append(curr_choice)

In [ ]:
for fish in files:
    print(fish)
    plane_list = fish / "suite2p"
    planes = list(plane_list.glob("*00*"))
    
    for path in planes:
        # getting imaging data
        # loading suite2p data:
        # getting stimulus data
        
        if not (path / 'sensory_regressors.h5').exists():
            try:
                metadata_file = list(path.glob("*_metadata.json"))[0]

                with open(str(metadata_file), "r") as f:
                     metadata = json.load(f)
                stim = metadata["stimulus"]["log"]

                fs = int(metadata['imaging']['microscope_config']['scanning']['framerate'])

                dt_imaging = 1 / fs
                int_fact = 200
            except:
                print('aaaaaaaaaaaaaaaaaaaaaaaaa')
            try:
                suite2p_data = fl.load(path / 'data_from_suite2p_unfiltered.h5')
                traces = suite2p_data["traces"]

                len_rec = traces.shape[1]
            except:
                aligned = SplitDataset(fish / "aligned")
                len_rec = np.shape(aligned)[0]
            t_imaging = np.arange(len_rec)/fs


            pause_duration = stim[0]['duration'] * fs
            stim_duration = stim[1]['duration'] * fs


            n_options = num_rows*num_rows
            n_rep = [metadata['stimulus']['protocol']['receptive_fields']['v04_square_flashes_4x4']['n_trials']][0]
            n_trials = (n_options * n_rep) 
            position_list = np.zeros((n_trials, 4))
            for_regs = np.zeros((n_options, n_trials * 2 + 1))

            regs = np.zeros((64, len_rec))
            t1 = pause_duration

            for i in range(1, n_trials * 2, 2):
                curr_trial = stim[i]['clip_mask']
                position_list[(i//2) - 1, :] = curr_trial

                for j in range(n_options):
                    if curr_trial == choices[j]:
                        for_regs[j, i-1] = 1
                        regs[j, t1:(t1 + stim_duration)] = 1

                t1 = t1 + stim_duration + pause_duration 

            tau_fs = REGRESSOR_TAU_S * fs
            kernel = np.exp(-np.arange(1000) / tau_fs)
            t_imaging_int = np.arange(len_rec*int_fact)*dt_imaging/int_fact


            regs_conv = np.zeros((n_options, len_rec))

            try:
                num_traces, len_rec = np.shape(traces)
                regs_vals = np.zeros((n_options, num_traces))

                for i in range(n_options):
                    regs_conv[i] = np.convolve(regs[i], kernel)[:np.shape(traces)[1]]

                    tmp_reg_vals = np.dot(traces, regs_conv[i]) - num_traces * np.mean(traces, 1) * np.mean(regs_conv[i])
                    tmp_reg_vals /= (traces.shape[1] - 1) * np.std(traces, 1) * np.std(regs_conv[i])
                    regs_vals[i] = tmp_reg_vals



                d = {'regressors': regs,
                     'regressors_conv': regs_conv,
                     'regressors_values': regs_vals,
                }
                fl.save(path / 'sensory_regressors.h5', d)

            except:
                for i in range(n_options):
                    regs_conv[i] = np.convolve(regs[i], kernel)[:np.shape(traces)[1]]

                d = {'regressors': regs,
                     'regressors_conv': regs_conv,
                }

                fl.save(path / 'sensory_regressors_partial.h5', d)

In [ ]:
fig, ax = plt.subplots(1,1)
ax.plot(regs[0:4].T)